# Tactical Corridor Synthesis (S3)

This notebook synthesises the **amber tactical corridor** from the city-core route
candidates: which segments carry it, how its intensity should ramp toward the gateway,
and which segments are physically treatable versus where it meets a road and becomes a
**crossing point**.

Scope (per the pavilion-gateway model): the amber corridor serves the **city-core**
approaches into the Ryder Street gateway - Colmore Row, New Street, Moor Street and
(experimental) Snow Hill. The **Nechells/north** side is handled by the crossing +
wayfinders, not the corridor, so it is excluded here.

Descriptive only, on the Gate 0B audited network. Weights and thresholds are documented
assumptions; segment widths/surfaces are OSM proxies pending survey. No funding claim.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, ast, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, geopandas as gpd, networkx as nx, osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import LineString

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))
from spinelens.spatial import audit
from spinelens.metrics import legibility as lg
from spinelens.models import corridor as co

DATA = PHASE1_ROOT / "data"
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "corridor_media"
TABLES = PHASE1_ROOT / "outputs" / "tables"
EXPORTS = PHASE1_ROOT / "outputs" / "exports"
REPORTS = PHASE1_ROOT / "outputs" / "reports"
for d in (FIG_DIR, TABLES, EXPORTS, REPORTS): d.mkdir(parents=True, exist_ok=True)

G = ox.load_graphml(DATA / "raw" / "osm_network" / "gate0b_osm_walk_graph.graphml")
und = G.to_undirected(); largest = max(nx.connected_components(und), key=len)
coords = {n: (float(d["y"]), float(d["x"])) for n, d in G.nodes(data=True)}
cl = {n: c for n, c in coords.items() if n in largest}

def _hwlist(h):
    if isinstance(h, list): return h
    if isinstance(h, str) and h.startswith("["):
        try: return ast.literal_eval(h)
        except (ValueError, SyntaxError): return [h]
    return [h]
def rep_hw(h):
    hs = [str(x) for x in _hwlist(h)]
    for c in hs:
        if c in co.CROSSING_HIGHWAY: return c
    if "steps" in hs: return "steps"
    return hs[0]

node_severity = {n: 0.0 for n in G.nodes}
for u, v, d in G.edges(data=True):
    s = max((lg.crossing_severity(x) for x in _hwlist(d.get("highway"))), default=0.0)
    if s: node_severity[u] = max(node_severity[u], s); node_severity[v] = max(node_severity[v], s)

nodes = pd.read_csv(DATA / "route_nodes_phase1.csv").set_index("node_id")
fams = pd.read_csv(DATA / "route_families_phase1.csv")
CITY_CORE = {"colmore_row", "new_street_station", "moor_street_queensway", "snow_hill_station"}
inbound = fams[(fams["gateway_node_id"] == "ryder_street_pavilion_search_area")
               & (fams["origin_node_id"].isin(CITY_CORE))]
def snap(nid): return audit.nearest_node(cl, (float(nodes.loc[nid,"latitude"]), float(nodes.loc[nid,"longitude"])))[0]
routes = {f.route_family_id.replace("_to_ryder_gateway", ""): nx.shortest_path(G, snap(f.origin_node_id), snap(f.gateway_node_id), weight="length")
          for _, f in inbound.iterrows()}
g_node = snap("ryder_street_pavilion_search_area")
print("city-core corridor routes:", list(routes))

## Segment sharing, intensity and suitability

In [ ]:
# undirected edge geometry / highway / length lookup
edges = ox.graph_to_gdfs(G, nodes=False)
edge_geom, edge_hw, edge_len = {}, {}, {}
for (u, v, k), row in edges.iterrows():
    key = co.edge_key(u, v)
    if key not in edge_geom:
        edge_geom[key] = row.geometry if row.geometry is not None else LineString([(coords[u][1], coords[u][0]), (coords[v][1], coords[v][0])])
        edge_hw[key] = rep_hw(row.get("highway"))
        edge_len[key] = float(row.get("length", 0.0))

mult = co.segment_multiplicity(routes)
rows = []
for key, users in mult.items():
    geom = edge_geom.get(key) or LineString([(coords[key[0]][1], coords[key[0]][0]), (coords[key[1]][1], coords[key[1]][0])])
    mid = geom.interpolate(0.5, normalized=True)
    dist_gw = audit.haversine_m((mid.y, mid.x), coords[g_node])
    sev = max(node_severity.get(key[0], 0.0), node_severity.get(key[1], 0.0))
    hw = edge_hw.get(key, "unclassified")
    suit = co.segment_suitability(hw, sev)
    rows.append({"key": key, "geometry": geom, "multiplicity": len(users),
                 "routes": ",".join(sorted(users)), "dist_to_gateway_m": round(dist_gw, 0),
                 "highway": hw, "severity": sev, "role": suit["role"], "suitability": suit["score"],
                 "length_m": round(edge_len.get(key, 0.0), 1)})
seg = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326)

# intensity: blend route-sharing with nearness to gateway (deck: ramp up near gateway)
mult_frac = co.normalize(seg["multiplicity"].tolist())
prox_frac = co.normalize([-d for d in seg["dist_to_gateway_m"]])  # closer -> higher
seg["intensity"] = [round(co.corridor_intensity(m, p), 3) for m, p in zip(mult_frac, prox_frac)]
display(seg.drop(columns="geometry").sort_values(["multiplicity", "intensity"], ascending=False).head(12))

## Corridor structure and crossing points

In [ ]:
summary = co.corridor_summary(mult)
trunk = seg[seg["multiplicity"] >= 2]; spur = seg[seg["multiplicity"] == 1]

# Major-road severance check (SPATIAL): how close does the city-core spine get to any
# trunk/primary/secondary road? Pedestrian routes cross/encounter major roads at points,
# so we measure each interior route node's distance to the nearest major-road geometry.
major = edges[edges["highway"].apply(lambda h: any(str(x) in co.CROSSING_HIGHWAY
              for x in (h if isinstance(h, list) else [h])))].to_crs(27700)
maj_union = major.geometry.union_all()
route_nodes_all = {}
for nm, path in routes.items():
    for n in path[1:-1]:
        route_nodes_all.setdefault(n, set()).add(nm)
import geopandas as _gpd
from shapely.geometry import Point as _Point
rn = list(route_nodes_all)
rn_m = _gpd.GeoSeries([_Point(coords[n][1], coords[n][0]) for n in rn], crs=4326).to_crs(27700)
node_major_dist = {n: float(rn_m.iloc[i].distance(maj_union)) for i, n in enumerate(rn)}
CROSS_THRESHOLD_M = 30.0
xrows = [{"node": n, "lat": coords[n][0], "lon": coords[n][1],
          "dist_to_major_road_m": round(node_major_dist[n], 1),
          "routes": ",".join(sorted(route_nodes_all[n])),
          "dist_to_gateway_m": round(audit.haversine_m(coords[n], coords[g_node]), 0)}
         for n in rn if node_major_dist[n] <= CROSS_THRESHOLD_M]
crossings = pd.DataFrame(xrows).sort_values("dist_to_gateway_m") if xrows else pd.DataFrame(
    columns=["node","lat","lon","dist_to_major_road_m","routes","dist_to_gateway_m"])
min_major = round(min(node_major_dist.values()), 1)
median_major = round(float(pd.Series(node_major_dist).median()), 1)

print(f"corridor segments: {summary['segments']} | trunk (shared): {summary['trunk_segments']} "
      f"| spur: {summary['spur_segments']} | max routes on a segment: {summary['max_multiplicity']}")
print(f"trunk length: {trunk['length_m'].sum()/1000:.2f} km | spur length: {spur['length_m'].sum()/1000:.2f} km")
print(f"city-core route nodes within {CROSS_THRESHOLD_M:.0f} m of a major road: {len(crossings)} "
      f"(nearest major road {min_major} m, median {median_major} m)")

cn = seg[seg["routes"].str.contains("colmore") & seg["routes"].str.contains("new_street")]
print(f"\nColmore & New Street share {len(cn)} segments ({cn['length_m'].sum():.0f} m) -> "
      f"{'they merge into a common spine' if len(cn) else 'no shared spine found'}.")

seg.drop(columns="key").to_file(EXPORTS / "tactical_corridor_segments.geojson", driver="GeoJSON")
seg.drop(columns=["key", "geometry"]).to_csv(TABLES / "tactical_corridor_segment_scores.csv", index=False)
if len(crossings):
    crossings.to_csv(TABLES / "tactical_corridor_crossing_points.csv", index=False)
print("exported corridor segments geojson + segment scores csv")

## Visuals

In [ ]:
seg_m = seg.to_crs(27700)
def to_m(n):
    from shapely.geometry import Point
    return gpd.GeoSeries([Point(coords[n][1], coords[n][0])], crs=4326).to_crs(27700).iloc[0]
base = ox.graph_to_gdfs(G, nodes=False).to_crs(27700)
minx, miny, maxx, maxy = seg_m.total_bounds; pad = 120
gm = to_m(g_node)

# FIGURE 1 - corridor intensity (amber), width by route multiplicity
fig, ax = plt.subplots(figsize=(11, 9))
base.cx[minx-pad:maxx+pad, miny-pad:maxy+pad].plot(ax=ax, color="#ececec", linewidth=0.4, zorder=1)
seg_m.plot(ax=ax, column="intensity", cmap="autumn_r", linewidth=1.5 + 2.5*np.array(mult_frac),
           zorder=3, legend=True, legend_kwds={"label": "corridor intensity", "shrink": 0.6})
for nm, path in routes.items():
    o = to_m(path[0]); ax.scatter(o.x, o.y, s=70, color="#1a73e8", edgecolor="white", zorder=5)
    ax.annotate(nm, (o.x, o.y), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.scatter(gm.x, gm.y, marker="*", s=420, color="black", edgecolor="white", zorder=6, label="Ryder gateway")
ax.set_xlim(minx-pad, maxx+pad); ax.set_ylim(miny-pad, maxy+pad); ax.set_aspect("equal")
ax.legend(loc="upper left"); ax.set_title("Tactical corridor: intensity (amber) and route sharing (line width)")
fig.savefig(FIG_DIR / "figK_corridor_intensity.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIGURE 2 - suitability: treatable surface + crossing points (node-level)
role_color = {"corridor_surface": "#34a853", "crossing_required": "#d93025",
              "unsuitable_access": "#000000", "review": "#9aa0a6"}
fig, ax = plt.subplots(figsize=(11, 9))
base.cx[minx-pad:maxx+pad, miny-pad:maxy+pad].plot(ax=ax, color="#ececec", linewidth=0.4, zorder=1)
for role, grp in seg_m.groupby("role"):
    grp.plot(ax=ax, color=role_color.get(role, "#9aa0a6"), linewidth=3, zorder=3, label=role)
if len(crossings):
    cx = gpd.GeoSeries(gpd.points_from_xy(crossings["lon"], crossings["lat"], crs=4326)).to_crs(27700)
    ax.scatter(cx.x, cx.y, s=180, facecolor="none", edgecolor="#d93025", linewidth=2.0, zorder=5,
               label="crossing point (meets major road)")
ax.scatter(gm.x, gm.y, marker="*", s=420, color="black", edgecolor="white", zorder=6, label="Ryder gateway")
ax.set_xlim(minx-pad, maxx+pad); ax.set_ylim(miny-pad, maxy+pad); ax.set_aspect("equal")
ax.legend(loc="upper left"); ax.set_title("Corridor suitability: treatable surface and crossing points")
fig.savefig(FIG_DIR / "figL_corridor_suitability.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# FIGURE 3 - intensity ramp toward gateway + trunk/spur split
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(seg["dist_to_gateway_m"], seg["intensity"], c=seg["multiplicity"], cmap="viridis")
axes[0].set_xlabel("distance to gateway (m)"); axes[0].set_ylabel("corridor intensity")
axes[0].set_title("Intensity ramps up approaching the gateway")
axes[0].invert_xaxis()
length_by_mult = seg.groupby("multiplicity")["length_m"].sum() / 1000
axes[1].bar(length_by_mult.index.astype(str), length_by_mult.values, color="#f59e0b")
axes[1].set_xlabel("routes sharing the segment"); axes[1].set_ylabel("corridor length (km)")
axes[1].set_title("Corridor length: shared trunk vs single-route spurs")
fig.tight_layout(); fig.savefig(FIG_DIR / "figM_corridor_profile.png", dpi=130, bbox_inches="tight"); plt.show()

## Findings note

In [ ]:
from spinelens.gate0b import utc_now_iso
ts = utc_now_iso()
lines = [
    "# Tactical Corridor Synthesis Note (S3)",
    "", f"Generated: {ts}. City-core corridor on the Gate 0B audited network. Descriptive only.",
    "", "## Structure", "",
    f"- Routes synthesised: {', '.join(routes)}.",
    f"- Corridor segments: {summary['segments']} ({summary['trunk_segments']} shared trunk, {summary['spur_segments']} spur).",
    f"- Shared trunk length: {trunk['length_m'].sum()/1000:.2f} km; spur length: {spur['length_m'].sum()/1000:.2f} km.",
    f"- Max routes on a single segment: {summary['max_multiplicity']}.",
    "", "## Deck questions answered (descriptively)", "",
    f"- **Colmore + New Street merge?** They share {len(cn)} segments (~{cn['length_m'].sum():.0f} m) -> "
    f"{'yes, a common spine forms' if len(cn) else 'no shared spine'}.",
    "- **One spine vs many?** The shared trunk concentrates near the gateway; origins arrive on spurs that "
    "fold into it - spurs feeding one trunk, not parallel corridors.",
    "- **Intensity ramp?** Intensity is highest where multiplicity is high and near the gateway (figK/figM).",
    "", "## KEY FINDING: the city-core spine is largely severance-free", "",
    f"- City-core route nodes within {int(CROSS_THRESHOLD_M)} m of a major road: **{len(crossings)}**.",
    f"- Nearest major road to the spine: **{min_major} m** (median {median_major} m), despite "
    f"{major.geometry.length.sum()/1000:.0f} km of major roads in the study extract.",
    "- Interpretation: the city-core approaches run on footways/minor streets and do not cross major",
    "  carriageways, so the amber corridor can be a near-continuous treatable surface from the city core",
    "  to the gateway. The major-road severance is concentrated on the **Nechells/Dartmouth side**",
    "  (handled by the Gate 0C crossing), consistent with the pavilion-gateway model.",
    "- Budget implication: the city-core corridor is mostly low-cost surface treatment; the costly",
    "  physical barrier is the single Gate 0C crossing, not the corridor.",
    "", "## Caveats", "",
    "- Segment widths/surfaces are OSM proxies; on-site survey needed before design.",
    "- 'Severance-free' is from OSM road geometry; verify footway quality/continuity on site.",
    "- Intensity weights are documented assumptions, subject to sensitivity testing.",
    "- Snow Hill is experimental/low-priority (least legible inbound route); shown for completeness.",
    "- Anchors provisional; OSM volunteered data pending OS cross-check; no funding claim.",
]
(REPORTS / "tactical_corridor_note.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:24]))

## What this unlocks

A defined city-core amber spine (shared trunk + feeder spurs), an intensity gradient
ramping to the gateway, and the explicit crossing points along it - each of which can be
evidence-checked like Gate 0C. This is the backbone the digital wayfinder content (S7)
sits on, and a key input to the consolidated budget pack (S8).